# 😊 Sentiment Analysis Model

Train a classifier to detect review sentiment: Positive, Negative, or Neutral.

**Algorithms**: Naive Bayes (recommended), Logistic Regression
**Input**: Review text
**Output**: Sentiment label (Positive/Negative/Neutral)


In [ ]:
# Install required libraries
!pip install -q scikit-learn pandas numpy nltk matplotlib seaborn joblib

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import joblib
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('punkt')
nltk.download('stopwords')

print('✅ All libraries imported successfully')

## 📊 Load Training Data

Dataset should have: `review` and `sentiment` columns (0=Negative, 1=Neutral, 2=Positive)

In [ ]:
# Sample training data
sample_data = {
    'review': [
        'Love this product!',                        # Positive
        'Amazing quality',                           # Positive
        'Terrible, waste of money',                 # Negative
        'Dont like it at all',                       # Negative
        'It is okay, nothing special',               # Neutral
        'Average product',                           # Neutral
        'Best purchase ever!',                       # Positive
        'Very disappointing',                        # Negative
        'Good value for money',                      # Positive
        'Has some pros and cons',                    # Neutral
    ],
    'sentiment': [2, 2, 0, 0, 1, 1, 2, 0, 2, 1]  # 0=Negative, 1=Neutral, 2=Positive
}

df = pd.DataFrame(sample_data)
print('Dataset shape:', df.shape)
print('\nSentiment distribution:')
sentiment_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
print(df['sentiment'].map(sentiment_map).value_counts())
print('\nSample data:')
print(df.head())

## 🧹 Data Preprocessing

In [ ]:
import re

def preprocess_text(text):
    """Clean and normalize text"""
    if not isinstance(text, str):
        return ''
    
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['review_clean'] = df['review'].apply(preprocess_text)
print('✅ Preprocessing complete')
print('\nCleaned samples:')
print(df[['review', 'review_clean']].head())

## 🔢 Feature Vectorization

In [ ]:
vectorizer = TfidfVectorizer(max_features=100, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['review_clean'])
y = df['sentiment']

print(f'✅ Vectorization complete')
print(f'Feature matrix shape: {X.shape}')

## 📚 Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'✅ Train-test split complete')
print(f'Training set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')

## 🤖 Train Models

In [ ]:
# 1. Naive Bayes (Recommended)
print('Training Naive Bayes...')
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
nb_acc = accuracy_score(y_test, nb_pred)
print(f'✅ Naive Bayes Accuracy: {nb_acc:.4f}')

# 2. Logistic Regression
print('\nTraining Logistic Regression...')
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)
lr_acc = accuracy_score(y_test, lr_pred)
print(f'✅ Logistic Regression Accuracy: {lr_acc:.4f}')

## 📊 Model Evaluation

In [ ]:
print('='*60)
print('NAIVE BAYES - PRIMARY MODEL')
print('='*60)
print('\nClassification Report:')
print(classification_report(y_test, nb_pred, target_names=['Negative', 'Neutral', 'Positive']))

print('\nConfusion Matrix:')
cm = confusion_matrix(y_test, nb_pred)
print(cm)

# Model comparison
print('\n' + '='*60)
print('MODEL COMPARISON')
print('='*60)
models = pd.DataFrame({
    'Model': ['Naive Bayes', 'Logistic Regression'],
    'Accuracy': [nb_acc, lr_acc]
})
models = models.sort_values('Accuracy', ascending=False)
print(models.to_string(index=False))

## 💾 Save Model

In [ ]:
joblib.dump(nb_model, 'sentiment_model.pkl')
print('✅ Sentiment model saved!')
print('   - sentiment_model.pkl')
print('\nPlace in: backend/models/trained_models/')

## 🧪 Test the Model

In [ ]:
test_reviews = [
    'This product is amazing!',
    'Terrible quality, very upset',
    'Its okay, nothing great'
]

sentiment_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}

print('Testing model on new reviews:')
print('='*60)

for review in test_reviews:
    review_clean = preprocess_text(review)
    X_new = vectorizer.transform([review_clean])
    pred = nb_model.predict(X_new)[0]
    confidence = max(nb_model.predict_proba(X_new)[0])
    
    print(f'Review: {review}')
    print(f'Sentiment: {sentiment_map[pred]} (Confidence: {confidence:.2%})')
    print('-'*60)